# Lab 2 · Clean, connect and check

**In class:** Part A, guided, on a messy version of the penguins data · **At home:** Part B, the same steps on your own two tables
**Due:** Sunday, November 8 at 11:59 PM (extended for Election Day) · **Submit:** your repo URL on CourseWorks

Tonight's three skills:
1. **Clean** a messy CSV with pandas and load it into DuckDB
2. **Connect** two tables with a join, on a key you've checked
3. **Check** every join by counting rows before and after

Run cells top to bottom with **Shift+Enter**.

---
# Part A · Guided: messy penguins + a lookup table

## A1. Read the messy file with pandas

`penguins_raw.csv` is the same study as last week's penguins, in its original form: long column names, extra fields, comments. **pandas** is a Python library for working with tables; a table in pandas is called a **DataFrame**.

In [339]:
from pathlib import Path
import pandas as pd
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
raw = pd.read_csv(ROOT / "data" / "penguins_raw.csv")

print(raw.shape)   # (rows, columns)
raw.head()

(344, 17)


,studyName,Sample Number,Species,Region,Island,Stage,Individual ID,Clutch Completion,Date Egg,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,Delta 15 N (o/oo),Delta 13 C (o/oo),Comments
0,PAL0708,1,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A1,Yes,2007-11-11,39.1,18.7,181.0,3750.0,MALE,NaN,NaN,Not enough blood for isotopes.
1,PAL0708,2,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A2,Yes,2007-11-11,39.5,17.4,186.0,3800.0,FEMALE,8.94956,-24.69454,NaN
2,PAL0708,3,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A1,Yes,2007-11-16,40.3,18.0,195.0,3250.0,FEMALE,8.36821,-25.33302,NaN
3,PAL0708,4,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A2,Yes,2007-11-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Adult not sampled.
4,PAL0708,5,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N3A1,Yes,2007-11-16,36.7,19.3,193.0,3450.0,FEMALE,8.76651,-25.32426,NaN


How much is missing in each column? `isna()` marks missing values; `.sum()` counts them per column.

In [340]:
raw.isna().sum()

studyName                0
Sample Number            0
Species                  0
Region                   0
Island                   0
Stage                    0
Individual ID            0
Clutch Completion        0
Date Egg                 0
Culmen Length (mm)       2
Culmen Depth (mm)        2
Flipper Length (mm)      2
Body Mass (g)            2
Sex                     11
Delta 15 N (o/oo)       14
Delta 13 C (o/oo)       13
Comments               290
dtype: int64

Look at the species column. What would make it hard to use as a key?

In [341]:
raw["Species"].value_counts()

Species
Adelie Penguin (Pygoscelis adeliae)          152
Gentoo penguin (Pygoscelis papua)            124
Chinstrap penguin (Pygoscelis antarctica)     68
Name: count, dtype: int64

**Notice:** long names with the Latin name in parentheses, and inconsistent capitals ("Adelie **P**enguin" vs. "Gentoo **p**enguin"). Text that *looks* the same to a person often won't match exactly in a join.

## A2. Three cleaning steps

**Step 1. Rename columns** to short, lowercase names with no spaces, so they're easy to type in SQL.

In [342]:
penguins = raw.rename(columns={
    "studyName": "study",
    "Individual ID": "individual_id",
    "Species": "species_raw",
    "Island": "island",
    "Date Egg": "date_egg",
    "Clutch Completion": "clutch_completion",
    "Culmen Length (mm)": "bill_length_mm",
    "Culmen Depth (mm)": "bill_depth_mm",
    "Flipper Length (mm)": "flipper_length_mm",
    "Body Mass (g)": "body_mass_g",
    "Sex": "sex",
})
penguins.columns.tolist()

['study',
 'Sample Number',
 'species_raw',
 'Region',
 'island',
 'Stage',
 'individual_id',
 'clutch_completion',
 'date_egg',
 'bill_length_mm',
 'bill_depth_mm',
 'flipper_length_mm',
 'body_mass_g',
 'sex',
 'Delta 15 N (o/oo)',
 'Delta 13 C (o/oo)',
 'Comments']

**Step 2. Standardize text.** Make `sex` lowercase, and pull the Latin name out of the parentheses into its own column. Scientific names are written the same way everywhere, which makes them a better key than the common name.

In [343]:
penguins["sex"] = penguins["sex"].str.lower()
penguins["latin_name"] = penguins["species_raw"].str.extract(r"\((.*)\)")   # text inside ( )

penguins[["species_raw", "latin_name", "sex"]].drop_duplicates().head(6)

,species_raw,latin_name,sex
0,Adelie Penguin (Pygoscelis adeliae),Pygoscelis adeliae,male
1,Adelie Penguin (Pygoscelis adeliae),Pygoscelis adeliae,female
3,Adelie Penguin (Pygoscelis adeliae),Pygoscelis adeliae,NaN
152,Gentoo penguin (Pygoscelis papua),Pygoscelis papua,female
153,Gentoo penguin (Pygoscelis papua),Pygoscelis papua,male
178,Gentoo penguin (Pygoscelis papua),Pygoscelis papua,NaN


**Step 3. Fix types and keep what you need.** The egg date came in as text; make it a real date. Then keep only the columns we'll use.

In [344]:
penguins["date_egg"] = pd.to_datetime(penguins["date_egg"])

penguins = penguins[[
    "study", "individual_id", "latin_name", "island", "date_egg", "clutch_completion",
    "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "sex",
]]
penguins.dtypes

study                           str
individual_id                   str
latin_name                      str
island                          str
date_egg             datetime64[us]
clutch_completion               str
bill_length_mm              float64
bill_depth_mm               float64
flipper_length_mm           float64
body_mass_g                 float64
sex                             str
dtype: object

## A3. Load it into DuckDB and verify with SQL

DuckDB can read a pandas DataFrame directly by its variable name. Always check the row count after moving data: **rows in should equal rows out.**

In [345]:
con = duckdb.connect(str(ROOT / "lab2.duckdb"))
con.sql("CREATE OR REPLACE TABLE penguins_clean AS SELECT * FROM penguins")

print("rows in pandas:", len(penguins))
con.sql("SELECT count(*) AS rows_in_duckdb FROM penguins_clean")

rows in pandas: 344


┌────────────────┐
│ rows_in_duckdb │
│     int64      │
├────────────────┤
│            344 │
└────────────────┘

In [346]:
con.sql("DESCRIBE penguins_clean")

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ study             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ individual_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ latin_name        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ island            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ date_egg          │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ clutch_completion │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ bill_length_mm    │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ bill_depth_mm     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ flipper_length_mm │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ body_mass_

## A4. What is one row? Is the ID really an ID?

There's a column called `individual_id`. If it identified one penguin, every penguin would appear once per season. Let's check.

In [347]:
con.sql("""
    SELECT count(*)                      AS rows,
           count(DISTINCT individual_id) AS distinct_ids
    FROM penguins_clean
""")

┌───────┬──────────────┐
│ rows  │ distinct_ids │
│ int64 │    int64     │
├───────┼──────────────┤
│   344 │          190 │
└───────┴──────────────┘

In [348]:
con.sql("""
    SELECT study, latin_name, island, individual_id
    FROM penguins_clean
    WHERE individual_id = 'N11A1'
""")

┌─────────┬────────────────────┬─────────┬───────────────┐
│  study  │     latin_name     │ island  │ individual_id │
│ varchar │      varchar       │ varchar │    varchar    │
├─────────┼────────────────────┼─────────┼───────────────┤
│ PAL0708 │ Pygoscelis adeliae │ Biscoe  │ N11A1         │
│ PAL0809 │ Pygoscelis papua   │ Biscoe  │ N11A1         │
└─────────┴────────────────────┴─────────┴───────────────┘

**The same ID belongs to an Adélie penguin in one season and a Gentoo the next.** It can't be one bird. It's most likely a nest-and-adult label the researchers reused. So:

- `individual_id` **alone is not a key.** A column named "ID" isn't automatically unique; always check with `count(DISTINCT ...)`.
- **One row = one adult penguin at one nest in one breeding season.** That's the table's **grain**, and you should be able to say it for every table you use.

## A5. A lookup table to join

`species_lookup.csv` is a small reference table: one row per species, with its common name, Latin name, genus and group.

In [349]:
con.sql(f"CREATE OR REPLACE TABLE species AS SELECT * FROM read_csv('{ROOT / 'data' / 'species_lookup.csv'}')")
con.sql("SELECT * FROM species")

┌───────────────────┬────────────────────────┬─────────────┬───────────────────────┐
│    common_name    │       latin_name       │    genus    │      group_name       │
│      varchar      │        varchar         │   varchar   │        varchar        │
├───────────────────┼────────────────────────┼─────────────┼───────────────────────┤
│ Adélie penguin    │ Pygoscelis adeliae     │ Pygoscelis  │ brush-tailed penguins │
│ Chinstrap penguin │ Pygoscelis antarcticus │ Pygoscelis  │ brush-tailed penguins │
│ Gentoo penguin    │ Pygoscelis papua       │ Pygoscelis  │ brush-tailed penguins │
│ Emperor penguin   │ Aptenodytes forsteri   │ Aptenodytes │ great penguins        │
└───────────────────┴────────────────────────┴─────────────┴───────────────────────┘

Before joining, check that the key is **unique** in the lookup table. If a key appeared twice, every matching penguin would be copied twice.

In [350]:
con.sql("""
    SELECT latin_name, count(*) AS n
    FROM species
    GROUP BY latin_name
    HAVING count(*) > 1
""")   # no rows back = the key is unique

┌────────────┬───────┐
│ latin_name │   n   │
│  varchar   │ int64 │
└────────────┴───────┘
        0 rows      

## A6. Join, and count what comes out

`JOIN ... ON` lines up rows where the key matches. We expect **344** rows out, one per penguin row.

In [351]:
con.sql("""
    SELECT count(*) AS rows_after_join
    FROM penguins_clean p
    JOIN species s ON p.latin_name = s.latin_name
""")

┌─────────────────┐
│ rows_after_join │
│      int64      │
├─────────────────┤
│             276 │
└─────────────────┘

**Only 276, not 344.** An inner `JOIN` silently drops rows that don't match. Nothing errored. This is a data cascade in the making.

## A7. Find what didn't match with a LEFT JOIN

A `LEFT JOIN` keeps every penguin row. Where there's no match, the lookup's columns come back empty (`NULL`), so we can find the problem rows.

In [352]:
con.sql("""
    SELECT p.latin_name, count(*) AS unmatched_rows
    FROM penguins_clean p
    LEFT JOIN species s ON p.latin_name = s.latin_name
    WHERE s.latin_name IS NULL
    GROUP BY p.latin_name
""")

┌───────────────────────┬────────────────┐
│      latin_name       │ unmatched_rows │
│        varchar        │     int64      │
├───────────────────────┼────────────────┤
│ Pygoscelis antarctica │             68 │
└───────────────────────┴────────────────┘

Compare with the lookup: the data says **Pygoscelis antarctic*a***, the lookup says **Pygoscelis antarctic*us***. Two sources spelled the same species differently, and 68 penguins vanished from the join.

## A8. Fix the key, then join again

Fix it at the source of the problem (the cleaning step), then reload and rejoin.

In [353]:
penguins["latin_name"] = penguins["latin_name"].replace(
    {"Pygoscelis antarctica": "Pygoscelis antarcticus"}
)
con.sql("CREATE OR REPLACE TABLE penguins_clean AS SELECT * FROM penguins")

con.sql("""
    SELECT count(*) AS rows_after_join
    FROM penguins_clean p
    JOIN species s ON p.latin_name = s.latin_name
""")   # now 344

┌─────────────────┐
│ rows_after_join │
│      int64      │
├─────────────────┤
│             344 │
└─────────────────┘

Which way you join matters. Start from the **lookup** with a `LEFT JOIN` to see species that have no penguins:

In [354]:
con.sql("""
    SELECT s.common_name, count(p.latin_name) AS penguin_rows
    FROM species s
    LEFT JOIN penguins_clean p ON p.latin_name = s.latin_name
    GROUP BY s.common_name
    ORDER BY penguin_rows DESC
""")

┌───────────────────┬──────────────┐
│    common_name    │ penguin_rows │
│      varchar      │    int64     │
├───────────────────┼──────────────┤
│ Adélie penguin    │          152 │
│ Gentoo penguin    │          124 │
│ Chinstrap penguin │           68 │
│ Emperor penguin   │            0 │
└───────────────────┴──────────────┘

## A9. What a duplicate key does

Here we fake a lookup table that lists Gentoo twice, then join. Watch the row count.

In [355]:
con.sql("""
    WITH species_with_dupe AS (
        SELECT * FROM species
        UNION ALL
        SELECT * FROM species WHERE common_name = 'Gentoo penguin'
    )
    SELECT count(*) AS rows_after_join
    FROM penguins_clean p
    JOIN species_with_dupe s ON p.latin_name = s.latin_name
""")   # 468: every Gentoo row was copied

┌─────────────────┐
│ rows_after_join │
│      int64      │
├─────────────────┤
│             468 │
└─────────────────┘

**344 became 468**, and every total or average involving Gentoos would now be wrong. That's why you check key uniqueness (A5) before every join.

## A10. Use the join to answer a question

Now that the tables connect, ask a question that needs both: the lookup's readable names, the penguin table's measurements.

In [356]:
con.sql("""
    SELECT s.common_name,
           p.island,
           count(*)                    AS penguins,
           round(avg(p.body_mass_g))   AS avg_mass_g
    FROM penguins_clean p
    JOIN species s ON p.latin_name = s.latin_name
    GROUP BY s.common_name, p.island
    ORDER BY s.common_name, p.island
""")

┌───────────────────┬───────────┬──────────┬────────────┐
│    common_name    │  island   │ penguins │ avg_mass_g │
│      varchar      │  varchar  │  int64   │   double   │
├───────────────────┼───────────┼──────────┼────────────┤
│ Adélie penguin    │ Biscoe    │       44 │     3710.0 │
│ Adélie penguin    │ Dream     │       56 │     3688.0 │
│ Adélie penguin    │ Torgersen │       52 │     3706.0 │
│ Chinstrap penguin │ Dream     │       68 │     3733.0 │
│ Gentoo penguin    │ Biscoe    │      124 │     5076.0 │
└───────────────────┴───────────┴──────────┴────────────┘

### The join checklist (use it in Part B)

1. **Grain:** what is one row in each table?
2. **Key:** which column connects them, and is it **unique** in the lookup side?
3. **Clean the key** the same way on both sides (spelling, capitals, spaces, codes vs. names, types).
4. **Count rows** before and after. More rows = a duplicate key. Fewer = unmatched keys.
5. **LEFT JOIN … IS NULL** to see exactly what didn't match, then decide: fix, drop, or keep as unknown.

---
# Part B · At home: your own two tables

Apply the checklist to the two tables you chose in Lab 1. If you'd like to swap datasets, that's fine; you just need two public tables that share a column.

## B1. Load both tables with pandas

In [357]:
FILE_A = "crashes.csv"     # <- your main table's file in data/
FILE_B = "speeding.csv"   # <- the related table's file in data/

a = pd.read_csv(ROOT / "data" / FILE_A)
b = pd.read_csv(ROOT / "data" / FILE_B)
print("A:", a.shape, "  B:", b.shape)
a.head()

A: (291998, 29)   B: (2697, 44)


,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,...,CONTRIBUTING FACTOR VEHICLE 2,CONTRIBUTING FACTOR VEHICLE 3,CONTRIBUTING FACTOR VEHICLE 4,CONTRIBUTING FACTOR VEHICLE 5,COLLISION_ID,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2,VEHICLE TYPE CODE 3,VEHICLE TYPE CODE 4,VEHICLE TYPE CODE 5
0,01/01/2022,7:33,NaN,NaN,40.771477,-73.91824,"(40.771477, -73.91824)",HOYT AVENUE NORTH,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4491400,Sedan,NaN,NaN,NaN,NaN
1,01/01/2022,4:30,NaN,NaN,NaN,NaN,NaN,Southern parkway,Jfk expressway,NaN,...,Unspecified,NaN,NaN,NaN,4491626,Sedan,Sedan,NaN,NaN,NaN
2,01/01/2022,7:57,NaN,NaN,NaN,NaN,NaN,WESTCHESTER AVENUE,SHERIDAN EXPRESSWAY,NaN,...,NaN,NaN,NaN,NaN,4491734,Sedan,NaN,NaN,NaN,NaN
3,01/01/2022,5:17,NaN,NaN,40.746930,-73.84866,"(40.74693, -73.84866)",GRAND CENTRAL PKWY,NaN,NaN,...,Unsafe Lane Changing,NaN,NaN,NaN,4491857,Sedan,Sedan,NaN,NaN,NaN
4,01/01/2022,1:30,NaN,NaN,40.819157,-73.96038,"(40.819157, -73.96038)",HENRY HUDSON PARKWAY,NaN,NaN,...,NaN,NaN,NaN,NaN,4491344,Sedan,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN


In [358]:
b.head()

,Unique Key,Created Date,Closed Date,Agency,Agency Name,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Additional Details,Location Type,Incident Zip,...,Vehicle Type,Taxi Company Borough,Taxi Pick Up Location,Bridge Highway Name,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Latitude,Longitude,Location
0,63576645,12/31/2024 05:47:24 PM,12/31/2024 06:29:44 PM,NYPD,New York City Police Department,Traffic,Chronic Speeding,NaN,Street/Sidewalk,11366,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.721959,-73.809697,POINT (-73.809696824262 40.721959131993)
1,63581854,12/31/2024 04:41:39 PM,12/31/2024 04:50:13 PM,NYPD,New York City Police Department,Traffic,Chronic Speeding,NaN,Street/Sidewalk,11366,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.721959,-73.809697,POINT (-73.809696824262 40.721959131993)
2,63578464,12/31/2024 01:29:55 PM,12/31/2024 01:45:38 PM,NYPD,New York City Police Department,Traffic,Chronic Speeding,NaN,Street/Sidewalk,11366,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.721959,-73.809697,POINT (-73.809696824262 40.721959131993)
3,63583710,12/31/2024 01:10:17 PM,12/31/2024 01:11:38 PM,NYPD,New York City Police Department,Traffic,Chronic Speeding,NaN,Street/Sidewalk,11366,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.721959,-73.809697,POINT (-73.809696824262 40.721959131993)
4,63576790,12/31/2024 01:10:11 PM,12/31/2024 01:11:49 PM,NYPD,New York City Police Department,Traffic,Chronic Speeding,NaN,Street/Sidewalk,11366,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.721959,-73.809697,POINT (-73.809696824262 40.721959131993)


**Grain** (double-click to edit). One row in table A is: a motor vehicle crash in NYC, as recorded by NYPD. One row in table B is: one 311 Service Request from an individual in NYC reporting "chronic speeding".

Issue is that I need both tables to be counts of crashes/requests per zip code to then join by Zip.

## B2. Clean: at least three steps

Use the Part A steps as a model: rename columns, standardize text, fix types, drop columns you don't need. Write each step in its own line with a comment saying *why*.

In [359]:
# Step 1: rename columns (short, lowercase, no spaces)
# a = a.rename(columns={"Country Code": "country_code", ...})

a = a.rename(columns={
    "CRASH DATE": "date",
    "CRASH TIME": "time",
    "ZIP CODE": "zip",
    "LATITUDE": "lat",
    "LONGITUDE": "long",
    "NUMBER OF PERSONS INJURED": "persons_injured",
    "NUMBER OF PERSONS KILLED": "persons_killed",
    "NUMBER OF PEDESTRIANS KILLED": "pedestrians_killed",
    "NUMBER OF CYCLIST KILLED": "cyclists_killed",
    "NUMBER OF MOTORIST KILLED": "motorists_killed",
})

b = b.rename(columns={
    "Created Date": "date",
    "Agency": "agency",
    "Problem Detail (formerly Descriptor)": "problem",
    "Incident Zip": "zip",
    "Borough": "borough",
    "Latitude": "lat",
    "Longitude": "long",
})

# Step 2: standardize the key text so both tables match
# a["country_code"] = a["country_code"].str.strip().str.upper()
# b["country_code"] = b["country_code"].str.strip().str.upper()

a.columns = a.columns.str.strip().str.lower().str.replace(" ", "_")
b.columns = b.columns.str.strip().str.lower().str.replace(" ", "_")

a.tail()

,date,time,borough,zip,lat,long,location,on_street_name,cross_street_name,off_street_name,...,contributing_factor_vehicle_2,contributing_factor_vehicle_3,contributing_factor_vehicle_4,contributing_factor_vehicle_5,collision_id,vehicle_type_code_1,vehicle_type_code_2,vehicle_type_code_3,vehicle_type_code_4,vehicle_type_code_5
291993,01/01/2025,18:57,NaN,NaN,40.798256,-73.827440,"(40.798256, -73.82744)",BRONX WHITESTONE BRIDGE,NaN,NaN,...,Following Too Closely,Following Too Closely,NaN,NaN,4790196,Sedan,Station Wagon/Sport Utility Vehicle,Sedan,NaN,NaN
291994,01/01/2025,18:20,NaN,NaN,40.796720,-73.976180,"(40.79672, -73.97618)",HENRY HUDSON PARKWAY,NaN,NaN,...,Unspecified,Unspecified,NaN,NaN,4783236,Station Wagon/Sport Utility Vehicle,Station Wagon/Sport Utility Vehicle,Sedan,NaN,NaN
291995,01/01/2025,14:29,BROOKLYN,11213.0,40.671870,-73.930980,"(40.67187, -73.93098)",NaN,NaN,202 UTICA AVE,...,NaN,NaN,NaN,NaN,4869426,Sedan,NaN,NaN,NaN,NaN
291996,01/01/2025,5:00,QUEENS,11419.0,NaN,NaN,NaN,NaN,NaN,103-16 VAN WYCK EXPY,...,NaN,NaN,NaN,NaN,4869536,Sedan,NaN,NaN,NaN,NaN
291997,01/01/2025,5:34,MANHATTAN,10002.0,40.720234,-73.984474,"(40.720234, -73.984474)",CLINTON ST,STANTON ST,NaN,...,NaN,NaN,NaN,NaN,4868969,Sedan,NaN,NaN,NaN,NaN


In [360]:
b.tail()

,unique_key,date,closed_date,agency,agency_name,problem_(formerly_complaint_type),problem,additional_details,location_type,zip,...,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,lat,long,location
2692,53048714,01/13/2022 09:44:04 AM,01/13/2022 10:04:15 AM,NYPD,New York City Police Department,Traffic,Chronic Speeding,NaN,Street/Sidewalk,10306,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.574900,-74.091094,POINT (-74.091093956568 40.574899555508)
2693,53044694,01/12/2022 10:44:01 AM,01/12/2022 11:59:13 AM,NYPD,New York City Police Department,Traffic,Chronic Speeding,NaN,Street/Sidewalk,11385,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.706116,-73.898494,POINT (-73.898494285801 40.706116351752)
2694,52983100,01/06/2022 06:50:34 AM,01/06/2022 07:08:44 AM,NYPD,New York City Police Department,Traffic,Chronic Speeding,NaN,Street/Sidewalk,11210,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.623534,-73.935699,POINT (-73.93569857784 40.623533542741)
2695,52963516,01/04/2022 06:16:05 PM,01/04/2022 07:17:07 PM,NYPD,New York City Police Department,Traffic,Chronic Speeding,NaN,Street/Sidewalk,11238,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.674004,-73.964872,POINT (-73.964872428323 40.674003511842)
2696,52947015,01/02/2022 04:09:00 PM,01/02/2022 04:42:20 PM,NYPD,New York City Police Department,Traffic,Chronic Speeding,NaN,Street/Sidewalk,11236,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.637547,-73.909278,POINT (-73.909278114582 40.637547097183)


In [361]:
## Step 3: fix a type, or drop rows/columns you don't need
# a["year"] = a["year"].astype(int)

#Checking data type for the date to remove "01/01/2025" in table A, so that both tables are the exact same range, 2022-2024.
print(a["date"].dtype, b["date"].dtype)

str str


In [362]:
#Removing "01/01/2025" from table A. Checking if any rows show that date.
a = a[a["date"] != "01/01/2025"].copy()

print((a["date"] == "01/01/2025").sum())

0


In [363]:
#Removed other columns. Checked data types.

a = a[[
    "date", "time", "zip", "lat", "long",
    "persons_injured", "persons_killed",
    "pedestrians_killed", "cyclists_killed", "motorists_killed",
    ]]

b = b[[
    "date", "agency", "problem", "zip",
    "borough", "lat", "long",
]]

a.dtypes


date                      str
time                      str
zip                   float64
lat                   float64
long                  float64
persons_injured         int64
persons_killed          int64
pedestrians_killed      int64
cyclists_killed         int64
motorists_killed        int64
dtype: object

In [364]:
b.dtypes

date           str
agency         str
problem        str
zip          int64
borough        str
lat        float64
long       float64
dtype: object

In [365]:
#From lab 1, I know a lot of zip codes were missing in table A.
a.isna().sum()

date                      0
time                      0
zip                   92100
lat                   23322
long                  23322
persons_injured           0
persons_killed            0
pedestrians_killed        0
cyclists_killed           0
motorists_killed          0
dtype: int64

In [366]:
%pip install geopandas

import geopandas as gpd
import pandas as pd

# The following work to convert coordinates to zip codes in table A was completed with the assistance of Claude (AI).

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [367]:
#  Modified Zip Code Tabulation Areas (MODZCTA) geoJSON shapefile came from NYC Open Data.
# Found at: https://data.cityofnewyork.us/Health/Modified-Zip-Code-Tabulation-Areas-MODZCTA-/pri4-ifjk/about_data

zips = gpd.read_file(ROOT / "data" / "MODZCTA.geojson")
print(zips.columns.tolist())

zip_col = next(c for c in zips.columns if c.lower() == "modzcta")
zips = (
    zips[[zip_col, "geometry"]]
    .rename(columns={zip_col: "zip_fill"})
    .to_crs("EPSG:4326")                            # same system as plain lat/long
)
zips["zip_fill"] = zips["zip_fill"].astype(str)

[':id', ':version', ':created_at', ':updated_at', 'modzcta', 'label', 'zcta', 'pop_est', 'geometry']


In [368]:
#Clean the existing zips to five digits and make the index unique

a = a.reset_index(drop=True)
a["zip"] = a["zip"].astype("string").str.extract(r"(\d{5})")[0]
print("zip missing before:", a["zip"].isna().sum())

#Picked out the rows that need filling: no zip, but usable coordinates.
lat = pd.to_numeric(a["lat"], errors="coerce")
lon = pd.to_numeric(a["long"], errors="coerce")
mask = a["zip"].isna() & lat.notna() & lon.notna() & (lat != 0) & (lon != 0)

pts = gpd.GeoDataFrame(
    a.loc[mask, []],                                       # keeps only the index
    geometry=gpd.points_from_xy(lon[mask], lat[mask]),     # x = longitude, y = latitude
    crs="EPSG:4326",
)

zip missing before: 92100


In [369]:
# Point-in-polygon join, then write the zips back. 
joined = gpd.sjoin(pts, zips, how="left", predicate="within")
joined = joined[~joined.index.duplicated(keep="first")]    # a border point can match twice

a.loc[joined.index, "zip"] = joined["zip_fill"]

#Knowing that 23322 were missing coordinates, I expect no more than that amount of zip codes to now be missing.
print("zip missing after:", a["zip"].isna().sum())

zip missing after: 20260


In [370]:
#Checking how many unique zip codes appear in table A.
a["zip"].value_counts()

zip
11207    5877
11208    3982
11236    3889
11234    3525
11368    3487
         ... 
11371       1
10152       1
10178       1
10171       1
10177       1
Name: count, Length: 227, dtype: int64[pyarrow]

In [371]:
#Checking how many unique zip codes appear in table B.
b["zip"].value_counts()

zip
11366    1436
11432     172
11420      72
10314      30
10468      24
         ... 
10464       1
10010       1
10470       1
10021       1
10027       1
Name: count, Length: 171, dtype: int64

In [372]:
# Aggregating each table to one row per zip.

a_by_zip = a.groupby("zip").agg(
    crashes=("date", "size"),
    persons_injured=("persons_injured", "sum"),
    persons_killed=("persons_killed", "sum"),
    pedestrians_killed=("pedestrians_killed", "sum"),
    cyclists_killed=("cyclists_killed", "sum"),
    motorists_killed=("motorists_killed", "sum"),
).reset_index()

a_by_zip["zip"] = a_by_zip["zip"].astype(int)

a_by_zip

,zip,crashes,persons_injured,persons_killed,pedestrians_killed,cyclists_killed,motorists_killed
0,10000,109,61,0,0,0,0
1,10001,1680,709,3,2,0,1
2,10002,2497,1273,11,10,0,1
3,10003,1239,657,2,2,0,0
4,10004,261,59,0,0,0,0
...,...,...,...,...,...,...,...
222,11693,356,189,2,1,0,1
223,11694,357,172,1,1,0,0
224,11695,1,0,0,0,0,0
225,11697,37,9,0,0,0,0


In [373]:
# Removing zips "10000" and "99999" from table A because they do not exist.
# I am choosing to drop this data because the location of the crash is not easily attributable to a zip code.
a_by_zip = a_by_zip[(a_by_zip["zip"] != 10000) & (a_by_zip["zip"] != 99999)].copy()

# Checking if any rows show that those two zips.
print("Rows with 10000 or 99999 as zip codes:", ((a_by_zip["zip"] == 10000)|(a_by_zip["zip"] == 99999)).sum())

print("Rows left:", len(a_by_zip))

Rows with 10000 or 99999 as zip codes: 0
Rows left: 225


In [374]:
multi = b.groupby("zip")["borough"].nunique()
print((multi > 1).sum(), "zips in 311 table have more than one borough")

b_by_zip = b.groupby("zip").agg(
    speeding_311=("date", "size"),
    borough=("borough", "first"),
).reset_index()

b_by_zip

0 zips in 311 table have more than one borough


,zip,speeding_311,borough
0,10001,1,MANHATTAN
1,10002,9,MANHATTAN
2,10003,1,MANHATTAN
3,10005,1,MANHATTAN
4,10006,1,MANHATTAN
...,...,...,...
166,11436,4,QUEENS
167,11691,12,QUEENS
168,11692,1,QUEENS
169,11693,2,QUEENS


## B3. Load into DuckDB and check the key

In [382]:
con = duckdb.connect(str(ROOT / "lab2.duckdb"))
con.sql("CREATE OR REPLACE TABLE table_a AS SELECT * FROM a_by_zip")
con.sql("CREATE OR REPLACE TABLE table_b AS SELECT * FROM b_by_zip")

KEY_A = "zip"   # <- after your renames
KEY_B = "zip"

# Is the key unique in table B? No rows back = unique.
con.sql(f"""
    SELECT "{KEY_B}", count(*) AS n
    FROM table_b
    GROUP BY "{KEY_B}"
    HAVING count(*) > 1
    LIMIT 10
""")

┌───────┬───────┐
│  zip  │   n   │
│ int64 │ int64 │
└───────┴───────┘
     0 rows    

## B4. Join and count

In [376]:
print("rows in table A:", con.sql("SELECT count(*) FROM table_a").fetchone()[0])

con.sql(f"""
    SELECT count(*) AS rows_after_join
    FROM table_a a
    JOIN table_b b ON a."{KEY_A}" = b."{KEY_B}"
""")

rows in table A: 225


┌─────────────────┐
│ rows_after_join │
│      int64      │
├─────────────────┤
│             171 │
└─────────────────┘

## B5. Find what didn't match

In [377]:
con.sql(f"""
    SELECT a."{KEY_A}", count(*) AS unmatched_rows
    FROM table_a a
    LEFT JOIN table_b b ON a."{KEY_A}" = b."{KEY_B}"
    WHERE b."{KEY_B}" IS NULL
    GROUP BY a."{KEY_A}"
    ORDER BY unmatched_rows DESC
""")

┌───────┬────────────────┐
│  zip  │ unmatched_rows │
│ int64 │     int64      │
├───────┼────────────────┤
│ 10004 │              1 │
│ 10179 │              1 │
│ 10020 │              1 │
│ 10030 │              1 │
│ 10037 │              1 │
│ 10039 │              1 │
│ 10040 │              1 │
│ 10041 │              1 │
│ 10044 │              1 │
│ 10045 │              1 │
│   ·   │              · │
│   ·   │              · │
│   ·   │              · │
│ 10279 │              1 │
│ 10281 │              1 │
│ 10282 │              1 │
│ 11005 │              1 │
│ 11040 │              1 │
│ 11251 │              1 │
│ 11359 │              1 │
│ 11371 │              1 │
│ 11695 │              1 │
│ 10106 │              1 │
└───────┴────────────────┘
  54 rows      2 columns
  (20 shown)             

**What didn't match, and what will you do about it?** (fix the key, drop those rows, or keep them as unknown, and why)
54 zip codes did not match, which was expected due to the difference between the row count of the two tables (225-171). I want to keep all rows because they can tell us there was a crash in a zip code (table a) but that no 311 speeding report was made in that same zip code (table b), or vice versa.



## B6. Answer one question that needs both tables

Write a query that joins the tables and uses columns from **both**, e.g. average GDP by region, complaints per 1,000 residents by ZIP code.

In [389]:
con.sql(f"""
    SELECT coalesce(a."{KEY_A}", b."{KEY_B}") AS zip,
        coalesce(b.speeding_311, 0) AS speeding_311,
        a.crashes,
        a.persons_killed,
        b.borough
    FROM table_a a
    FULL JOIN table_b b ON a."{KEY_A}" = b."{KEY_B}"
    ORDER BY speeding_311 DESC
    LIMIT 10
""")

┌───────┬──────────────┬─────────┬────────────────┬───────────────┐
│  zip  │ speeding_311 │ crashes │ persons_killed │    borough    │
│ int64 │    int64     │  int64  │     int64      │    varchar    │
├───────┼──────────────┼─────────┼────────────────┼───────────────┤
│ 11366 │         1436 │     553 │              2 │ QUEENS        │
│ 11432 │          172 │    2248 │              3 │ QUEENS        │
│ 11420 │           72 │    2011 │              5 │ QUEENS        │
│ 10314 │           30 │    3040 │              6 │ STATEN ISLAND │
│ 10468 │           24 │    2136 │              7 │ BRONX         │
│ 11367 │           22 │    1446 │              5 │ QUEENS        │
│ 10019 │           20 │    1845 │              3 │ MANHATTAN     │
│ 10007 │           19 │     495 │              1 │ MANHATTAN     │
│ 10011 │           18 │    1337 │              4 │ MANHATTAN     │
│ 11234 │           17 │    3525 │             11 │ BROOKLYN      │
└───────┴──────────────┴─────────┴──────────────

**What does the result tell you?** (one or two sentences)
Speeding reports to 311 do not necessarily correlate with crashes 1-to-1, but they do seem to at least relate to large numbers of crashes. One zip code in Queens seems to be an outlier for speeding reports to 311, while also not having the most crashes or deaths from vehicular collisions.

## B7. Submit

1. Run the cell below.
2. Answer the two reflection questions in `labs/lab2.md`.
3. Save this notebook (Cmd+S), then commit and push: **Source Control** → `Lab 2` → **Commit** → **Sync Changes**.
4. Paste your repo URL into the Lab 2 assignment on CourseWorks.

In [390]:
con.close()
print("Done. Save, commit, push, and submit your repo URL.")

Done. Save, commit, push, and submit your repo URL.
